<a href="https://colab.research.google.com/github/Indradumnabanerji/DataScienceTutorials/blob/main/json_reader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import json
import pandas as pd

class MyApplication:
    def __init__(self, commentary_config_path="commentary_config.json"):
        with open(commentary_config_path, 'r') as f:
            self.commentary_config = json.load(f)

        self.df_org_data = pd.DataFrame({"Org_Primary_ID": ["O1", "O2"], "Name": ["Company A", "Company B"]})
        self.df_product_data = pd.DataFrame({"Product_SKU_ID": ["P100", "P200"], "Description": ["Laptop", "Mouse"]})
        self.df_geo_data = pd.DataFrame({"Country_Code": ["US", "CA"], "Region": ["North America", "North America"]})
        self.df_customer_data = pd.DataFrame({"Customer_GUID": ["CUST1", "CUST2"], "CustName": ["Alice", "Bob"]})

        self.default_merge_col = "default_key"
        self.application_version = "1.2.3"

        self._dataframe_map = {
            "org_data_df": self.df_org_data, "product_data_df": self.df_product_data,
            "geo_data_df": self.df_geo_data, "customer_data_df": self.df_customer_data,
        }
        self._variable_map = {
            "default_merge_column_name": self.default_merge_col, "app_version": self.application_version,
            "org_id_col": self.commentary_config.get("hierarchies_meta", {}).get("Org", {}).get("id_col_name"),
            "product_id_col": self.commentary_config.get("hierarchies_meta", {}).get("Product", {}).get("id_col_name"),
            "geo_id_col": self.commentary_config.get("hierarchies_meta", {}).get("Geography", {}).get("id_col_name")
        }

    def _get_nested_value(self, data: dict, path: str):
        keys, current = path.split('.'), data
        for key in keys:
            if isinstance(current, dict) and key in current: current = current[key]
            else: return None
        return current

    def _resolve_placeholder(self, value):
        if not isinstance(value, str): return value
        if ':' not in value: return value

        prefix, key_or_path = value.split(':', 1)
        if prefix == "$df": return self._dataframe_map.get(key_or_path)
        elif prefix == "$cfg": return self._get_nested_value(self.commentary_config, key_or_path)
        elif prefix == "$var": return self._variable_map.get(key_or_path)
        return value

    def _traverse_and_interpret(self, structure):
        if isinstance(structure, dict):
            return {self._resolve_placeholder(k): self._traverse_and_interpret(v) for k, v in structure.items()}
        elif isinstance(structure, list):
            return [self._traverse_and_interpret(item) for item in structure]
        else: return self._resolve_placeholder(structure)

    def get_hierarchy_configs(self, hierarchy_config_path="hierarchy_config.json"):
        with open(hierarchy_config_path, 'r') as f:
            raw_configs = json.load(f)
        return self._traverse_and_interpret(raw_configs)

if __name__ == "__main__":
    commentary_cfg_data = {
        "hierarchies_meta": {
            "Org": {"id_col_hierarchy_type": "OrgType", "id_col_rename": "RenamedOrgID", "id_col_determined_value": "OrgDet", "id_col_merge": "OrgMergeKey", "id_col_name": "EmployeeID"},
            "Product": {"id_col_hierarchy_type": "ProdType", "id_col_rename": "RenamedProdID", "id_col_determined_value": "ProdDet", "id_col_merge": "ProdMergeKey", "id_col_name": "SKU"},
            "Geography": {"id_col_hierarchy_type": "GeoType", "id_col_rename": "RenamedGeoID", "id_col_determined_value": "GeoDet", "id_col_merge": "GeoMergeKey", "id_col_name": "CountryCode"}
        }, "general_app_settings": {"default_fallback": "N/A"}
    }
    with open("commentary_config.json", "w") as f: json.dump(commentary_cfg_data, f, indent=4)

    hierarchy_cfg_data = [
        {"name": "OrganizationHierarchy", "df": "$df:org_data_df", "hierarchy_type": "$cfg:hierarchies_meta.Org.id_col_hierarchy_type", "id_col": "$var:org_id_col", "rename_columns": {"$var:org_id_col": "$cfg:hierarchies_meta.Org.id_col_rename", "Determined Value": "$cfg:hierarchies_meta.Org.id_col_determined_value"}, "merge_on": "$cfg:hierarchies_meta.Org.id_col_merge"},
        {"name": "ProductHierarchy", "df": "$df:product_data_df", "hierarchy_type": "$cfg:hierarchies_meta.Product.id_col_hierarchy_type", "id_col": "$var:product_id_col", "rename_columns": {"$var:product_id_col": "$cfg:hierarchies_meta.Product.id_col_rename", "Determined Value": "$cfg:hierarchies_meta.Product.id_col_determined_value"}, "merge_on": "$cfg:hierarchies_meta.Product.id_col_merge"},
        {"name": "GeographyHierarchy", "df": "$df:geo_data_df", "hierarchy_type": "$cfg:hierarchies_meta.Geography.id_col_hierarchy_type", "id_col": "$var:geo_id_col", "rename_columns": {"$var:geo_id_col": "$cfg:hierarchies_meta.Geography.id_col_rename", "Determined Value": "$cfg:hierarchies_meta.Geography.id_col_determined_value"}, "merge_on": "$cfg:hierarchies_meta.Geography.id_col_merge"},
        {"name": "CustomConfigWithVariables", "df": "$df:customer_data_df", "hierarchy_type": "CustomType", "id_col": "$var:default_merge_column_name", "rename_columns": {"ID_From_App_Var": "$var:app_version", "FallbackValue": "$cfg:general_app_settings.default_fallback"}, "merge_on": "custom_static_key"},
        {"name": "HierarchyWithMissingRefs", "df": "$df:non_existent_df", "hierarchy_type": "$cfg:non_existent.path.type", "id_col": "$var:non_existent_var", "rename_columns": {"StaticKey": "StaticValue"}, "merge_on": "static_merge_key"}
    ]
    with open("hierarchy_config.json", "w") as f: json.dump(hierarchy_cfg_data, f, indent=4)

    app = MyApplication()
    final_resolved_configs = app.get_hierarchy_configs()

    print(f"Total hierarchies loaded: {len(final_resolved_configs)}")
    for i, config_item in enumerate(final_resolved_configs):
        print(f"\n--- Hierarchy {i+1}: '{config_item.get('name', 'Unnamed')}' ---")
        for key, value in config_item.items():
            if key == "df": print(f"  {key}: {type(value)} (ID: {id(value)})")
            elif isinstance(value, dict): print(f"  {key}:"); [print(f"    {sk}: {sv}") for sk, sv in value.items()]
            else: print(f"  {key}: {value}")

    print(f"\nIs 'OrganizationHierarchy' DF the same object as app.df_org_data? {final_resolved_configs[0]['df'] is app.df_org_data}")

Total hierarchies loaded: 5

--- Hierarchy 1: 'OrganizationHierarchy' ---
  name: OrganizationHierarchy
  df: <class 'pandas.core.frame.DataFrame'> (ID: 135466831610128)
  hierarchy_type: OrgType
  id_col: EmployeeID
  rename_columns:
    EmployeeID: RenamedOrgID
    Determined Value: OrgDet
  merge_on: OrgMergeKey

--- Hierarchy 2: 'ProductHierarchy' ---
  name: ProductHierarchy
  df: <class 'pandas.core.frame.DataFrame'> (ID: 135466010680720)
  hierarchy_type: ProdType
  id_col: SKU
  rename_columns:
    SKU: RenamedProdID
    Determined Value: ProdDet
  merge_on: ProdMergeKey

--- Hierarchy 3: 'GeographyHierarchy' ---
  name: GeographyHierarchy
  df: <class 'pandas.core.frame.DataFrame'> (ID: 135465809067792)
  hierarchy_type: GeoType
  id_col: CountryCode
  rename_columns:
    CountryCode: RenamedGeoID
    Determined Value: GeoDet
  merge_on: GeoMergeKey

--- Hierarchy 4: 'CustomConfigWithVariables' ---
  name: CustomConfigWithVariables
  df: <class 'pandas.core.frame.DataFrame'> (